# Single-device Tetris decision smoke and run

Select a Colab 2026.07 Python 3.12 runtime with one v5e TPU. Supply an explicit mounted persistent directory and upload the separate experiment source zip. This notebook installs the published PR3 commit named below and downloads the pinned public Base model. No training starts until you run the last cells.


In [ ]:
from pathlib import Path
TRAINING_REVISION = "REPLACE_WITH_PUBLISHED_PR3_COMMIT_SHA"
EXPERIMENT_ZIP = Path("/content/tetris-experiment-source.zip")
PERSISTENT_ROOT = Path("/content/drive/MyDrive/minifield-tetris-base")
assert len(TRAINING_REVISION) == 40 and "REPLACE" not in TRAINING_REVISION
assert EXPERIMENT_ZIP.is_file()
assert PERSISTENT_ROOT.parent.is_dir(), "Mount or configure persistent storage explicitly"
PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "jax[tpu]==0.7.2"], check=True)
package = f"minifield-training[numerical,storage,text] @ git+https://github.com/Minifield-Labs/minifield-training.git@{TRAINING_REVISION}"
subprocess.run([sys.executable, "-m", "pip", "install", package], check=True)
probe = "import jax; devices = jax.devices(); assert len(devices) == 1 and devices[0].platform == 'tpu', devices; print(jax.__version__, devices)"
subprocess.run([sys.executable, "-c", probe], check=True)


In [ ]:
from huggingface_hub import snapshot_download
MODEL_DIR = PERSISTENT_ROOT / "base-model"
snapshot_download(
    repo_id="LiquidAI/LFM2.5-230M-Base",
    revision="9d2be5519834990d30996f878b6771cccbd24f2c",
    allow_patterns=["config.json", "tokenizer.json", "model.safetensors"],
    local_dir=MODEL_DIR,
)


In [ ]:
import zipfile
EXPERIMENT_DIR = Path("/content/tetris-experiment")
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(EXPERIMENT_ZIP) as archive:
    archive.extractall(EXPERIMENT_DIR)
assert (EXPERIMENT_DIR / "tetris_experiment" / "train.py").is_file()
DATASET = PERSISTENT_ROOT / "expert-decisions.jsonl"
CHECKPOINTS = PERSISTENT_ROOT / "checkpoints"
RUN_ID = "tetris-base-expert-v1"


In [ ]:
if not DATASET.exists():
    subprocess.run([sys.executable, "-m", "tetris_experiment.prepare",
        "--model-dir", str(MODEL_DIR), "--output", str(DATASET),
        "--games", "80", "--max-ticks", "5000", "--seed", "17",
        "--sequence-length", "512"], cwd=EXPERIMENT_DIR, check=True)


The 2-update startup smoke verifies the real Base weight bytes, compiles the classifier on the TPU, commits finite updates, saves a full checkpoint, and plays a short learned-policy game. On rerun, it validates the latest complete checkpoint and skips the smoke when progress exists. The next cell resumes the latest complete state for up to 3 hours.


In [ ]:
subprocess.run([sys.executable, "-m", "tetris_experiment.train",
    "--model-dir", str(MODEL_DIR), "--dataset", str(DATASET),
    "--checkpoint-root", str(CHECKPOINTS), "--run-id", RUN_ID,
    "--platform", "tpu", "--resume-latest", "--skip-if-resumed",
    "--max-steps", "2",
    "--checkpoint-every", "2", "--report-every", "1",
    "--eval-games", "1", "--eval-max-ticks", "1000"],
    cwd=EXPERIMENT_DIR, check=True)


In [ ]:
subprocess.run([sys.executable, "-m", "tetris_experiment.train",
    "--model-dir", str(MODEL_DIR), "--dataset", str(DATASET),
    "--checkpoint-root", str(CHECKPOINTS), "--run-id", RUN_ID,
    "--resume-latest", "--platform", "tpu", "--max-hours", "3",
    "--checkpoint-every", "200", "--report-every", "10",
    "--eval-games", "2", "--eval-max-ticks", "2000"],
    cwd=EXPERIMENT_DIR, check=True)


Each checkpoint callback prints real-game `lines_per_game`, `pieces_per_game`, `survival_ticks`, and `completed_games`. Read `checkpoints/replays/step-XXXXXXXX-game-0.txt` for a tick-by-tick board and learned action trace. Each full FP32 state is about 2.75 GB; the run saves every 200 updates and at its final update, so provide enough persistent storage.
